## In-Class Lecture 32-1 - Flight Software Code Review

In this lecture, you are the last reviewer before a piece of flight software is approved. The program compiles cleanly with every course flag and passes its test. Your job is to use the tools from Reading 32-2 to find out what the test missed.

### Reviewing Opening your Environment on the Notre Dame Machines

<b>1.</b> Log into the Notre Dame Machines as you performed on Homework 1 using Tabby Terminal 

Open Tabby Terminal on your machine.

Type in: <code>ssh -Y username@student13.cse.nd.edu</code>
<ul><li>"username" is your NetID.</li>
<ul>
<li>Example: my email is matt.morrison@nd.edu, but my NetID is mmorri22, so I would enter:<p>
<code>ssh -Y mmorri22@student13.cse.nd.edu</code><p>
The <code>-Y</code> is included to treat the Notre Dame machines as a trusted client<p>
</li>
</ul>
</ul>

2. Next, we will create a directory where you will save your In-Class Code. First, we will go into the Dropbox directory where you will submit your code:

<code>cd esc-courses</code>

<code>cd fa26-cse-20133.01</code>

<code>cd dropbox</code>

Make a directory for you to practice your code for this lecture

    mkdir inclass32
    cd inclass32

Enable script so you can run the most up-to-date versions of C and C++:

    scl enable gcc-toolset-15 bash

> <b>Note:</b> If you use the <code>-std=c2x</code> and get the following error, this means you did not run the enable command<br>
> <code>gcc: error: unrecognized command line option ‘-std=c2x’; did you mean ‘-std=c1x’?</code>

To obtain the code for today's lecture, perform the following commands:

    wget https://raw.githubusercontent.com/mmorri22/cse20133/main/inclass/lec32/setup.sh
    chmod a+rx setup.sh
    ./setup.sh
    
To confirm, perform the <code>ls</code> command. You will see the following:

    corrupt.txt  full.txt  hot.txt  Makefile  nominal.txt  short.txt  telemetry.c

### In-Class Reading Discussion

<font color = "red">Class Introduction Question #1 - What is the difference between verification and validation? Which one failed for the Mars Climate Orbiter?</font>

<font color = "red">Class Introduction Question #2 - Using what you learned about binary fractions, explain why the Patriot missile's clock error grew the longer the system ran.</font>

<font color = "red">Class Introduction Question #3 - The Ariane 5 had two redundant guidance computers. Why did redundancy not save the mission?</font>

<font color = "red">Class Introduction Question #4 - What is undefined behavior, and why can a program with undefined behavior produce different results at <code>-O0</code> and <code>-O2</code>?</font>

<font color = "red">Class Introduction Question #5 - Example 3 and <code>off_by_one_runtime.c</code> contain the same bug. Why did the compiler catch one but not the other, and which tool caught the second?</font>

<font color = "red">Class Introduction Question #6 - What can <code>-fanalyzer</code> find that AddressSanitizer cannot, and what can AddressSanitizer find that <code>-fanalyzer</code> cannot?</font>

### The Program Under Review

<code>telemetry.c</code> processes temperature readings from a thruster nozzle sensor. Line 1 of each file is the number of readings the instrument <i>reports</i>; the rest are temperatures in degrees Celsius. The program computes the average and maximum, then packs the average into a 16-bit telemetry field in hundredths of a degree.

You have five test files. Each one represents something that can actually happen on a spacecraft:

<table>
    <tr><th>File</th><th>What it represents</th></tr>
    <tr><td><code>nominal.txt</code></td><td>Normal operation: 5 readings reported, 5 sent</td></tr>
    <tr><td><code>corrupt.txt</code></td><td>A corrupted header reports more readings than the buffer can hold</td></tr>
    <tr><td><code>full.txt</code></td><td>The instrument sends one more reading than it reported</td></tr>
    <tr><td><code>short.txt</code></td><td>A dropped packet: 6 readings reported, only 4 arrived</td></tr>
    <tr><td><code>hot.txt</code></td><td>A hot burn: every value is real and correct, just larger than usual</td></tr>
</table>

As you work, fill in the anomaly log at the bottom of this page.

### Step 1 - Run the Test

    make telemetry

The program builds with no warnings and prints the correct average for <code>nominal.txt</code>. Now run it on the other four files:

    ./telemetry corrupt.txt
    ./telemetry full.txt
    ./telemetry short.txt
    ./telemetry hot.txt

Nothing crashes. Look carefully at each output. Which ones would you trust? Would you sign off on this software?

### Step 2 - The Static Analyzer

    make analyze

Read the numbered events in the output. They describe a path through <code>process_telemetry</code>, step by step. Which input file sends the program down that path?

Fix the bug so that <code>make analyze</code> completes with no errors. There is more than one correct fix. Be ready to explain why yours is correct.

### Step 3 - AddressSanitizer

    make asan

This runs the program on <code>full.txt</code>. The sanitizer reports a <code>heap-buffer-overflow</code>. Look at the line that says how many bytes after the region the write landed, and connect it to the size of the <code>malloc</code>.

The program trusts the file to contain only as many values as the header says. Fix the read loop so that no file, however long, can write past the end of the buffer. Rerun <code>make asan</code> until it is clean.

### Step 4 - valgrind

    make valgrind

This runs an unoptimized build on <code>short.txt</code>. valgrind reports that the program makes decisions based on <i>uninitialised</i> values, and gives you line numbers.

Compare the average printed for <code>short.txt</code> to the four values in the file. Where did the wrong answer come from? Fix the bug, and decide what the program should <i>tell the operator</i> when the reported and actual counts disagree.

### Step 5 - The Bug No Tool Caught

Run the program on <code>hot.txt</code> one more time:

    ./telemetry hot.txt

The nozzle is reported at a <i>negative</i> 322 degrees. Every tool we have run so far is silent, because the cast <code>(int16_t)</code> tells the compiler you meant to do this.

<ul>
    <li>What is the largest temperature the 16-bit field can represent?</li>
    <li>How is this the same mistake that destroyed Ariane 5?</li>
</ul>

Fix it by checking the range <i>before</i> the conversion. If the value does not fit, the program should not silently produce a wrong one. Decide what it should do instead, and print a message that an operator on the ground would understand.

<b>Time Permitting:</b> Run <code>make floatcheck</code>. This sanitizer option would have caught the bug, but it is <i>not</i> included in <code>-fsanitize=undefined</code>. What does that tell you about relying on a single tool?

### Anomaly Log

Fill this in as you go. Professional flight software teams keep a record like this for every bug found during review.

<table>
    <tr><th>Bug</th><th>Input file that exposed it</th><th>Tool that caught it</th><th>Your fix, in one sentence</th></tr>
    <tr><td>1</td><td></td><td></td><td></td></tr>
    <tr><td>2</td><td></td><td></td><td></td></tr>
    <tr><td>3</td><td></td><td></td><td></td></tr>
    <tr><td>4</td><td></td><td></td><td></td></tr>
</table>

## Compilation Run

    make telemetry
    make analyze
    make asan
    make valgrind

When all four bugs are fixed, all four commands should complete with no errors, and every test file should produce either a correct answer or a clear error message.